### Supply Chain Optimization
**E-Commerce Demand Forecasting & Supply Chain Optimization**

Goals:
- Calculate Reorder Point (ROP) per item
- Compute Safety Stock using demand variability
- Apply Economic Order Quantity (EOQ) model
- Estimate stockout risk and inventory cost savings
- Generate actionable inventory recommendations

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
print('Libraries loaded')

Libraries loaded


In [2]:
# Cell 2 — Load Data
demand   = pd.read_csv('../data/processed/demand_clean.csv', parse_dates=['date'])
forecast = pd.read_csv('../data/processed/forecast_results.csv', parse_dates=['date'])
models   = pd.read_csv('../data/processed/model_comparison.csv')

print('Demand shape  :', demand.shape)
print('Forecast shape:', forecast.shape)
print()
print('=== Best Forecast Model ===')
best = models.sort_values('RMSE').iloc[0]
print(f"Model : {best['model']}")
print(f"RMSE  : {best['RMSE']}")
print(f"MAPE  : {best['MAPE']}%")
display(demand.head(3))

Demand shape  : (913000, 9)
Forecast shape: (90, 6)

=== Best Forecast Model ===
Model : XGBoost
RMSE  : 895.95
MAPE  : 1.99%


,date,store,item,sales,year,month,dow,week,quarter
0,2013-01-01,1,1,13,2013,1,1,1,1
1,2013-01-02,1,1,11,2013,1,2,1,1
2,2013-01-03,1,1,14,2013,1,3,1,1


In [3]:
# Cell 3 — Per-Item Demand Statistics
item_stats = demand.groupby('item').agg(
    avg_daily_demand = ('sales', 'mean'),
    std_daily_demand = ('sales', 'std'),
    max_daily_demand = ('sales', 'max'),
    min_daily_demand = ('sales', 'min'),
    total_sales      = ('sales', 'sum'),
    days_active      = ('sales', 'count')
).reset_index()

item_stats['cv'] = item_stats['std_daily_demand'] / item_stats['avg_daily_demand']  # Coefficient of variation

print('Item-level demand statistics computed')
print(item_stats.describe().round(2))
display(item_stats.head(5))

Item-level demand statistics computed
        item  avg_daily_demand  std_daily_demand  max_daily_demand  \
count  50.00             50.00             50.00              50.0   
mean   25.50             52.25             18.13             132.6   
std    14.58             21.53              6.89              50.6   
min     1.00             18.36              7.27              50.0   
25%    13.25             30.30             11.15              81.5   
50%    25.50             51.37             17.84             132.0   
75%    37.75             69.63             23.73             170.0   
max    50.00             88.03             29.52             231.0   

       min_daily_demand  total_sales  days_active     cv  
count             50.00        50.00         50.0  50.00  
mean               8.94    954090.24      18260.0   0.35  
std                5.64    393147.99          0.0   0.02  
min                0.00    335230.00      18260.0   0.34  
25%                3.25    553229.75

,item,avg_daily_demand,std_daily_demand,max_daily_demand,min_daily_demand,total_sales,days_active,cv
0,1,21.981599,8.468922,59,1,401384,18260,0.385273
1,2,58.574151,20.093015,150,9,1069564,18260,0.343036
2,3,36.642223,13.179441,104,7,669087,18260,0.359679
3,4,22.010241,8.403898,66,0,401907,18260,0.381818
4,5,18.358708,7.265167,50,1,335230,18260,0.395734


In [4]:
# Cell 4 — Safety Stock Calculation
# Safety Stock = Z * std_demand * sqrt(lead_time)
# Z = 1.645 for 95% service level

LEAD_TIME    = 7    # days (supplier lead time)
SERVICE_LEVEL = 0.95
Z_SCORE      = stats.norm.ppf(SERVICE_LEVEL)  # 1.645

item_stats['safety_stock'] = (
    Z_SCORE * item_stats['std_daily_demand'] * np.sqrt(LEAD_TIME)
).round(2)

print(f'Service Level  : {SERVICE_LEVEL*100:.0f}%')
print(f'Z-Score        : {Z_SCORE:.3f}')
print(f'Lead Time      : {LEAD_TIME} days')
print()
print('Safety Stock stats:')
print(item_stats['safety_stock'].describe().round(2))

Service Level  : 95%
Z-Score        : 1.645
Lead Time      : 7 days

Safety Stock stats:
count     50.00
mean      78.91
std       29.97
min       31.62
25%       48.52
50%       77.64
75%      103.29
max      128.48
Name: safety_stock, dtype: float64


In [5]:
# Cell 5 — Reorder Point (ROP)
# ROP = (avg_daily_demand * lead_time) + safety_stock

item_stats['rop'] = (
    item_stats['avg_daily_demand'] * LEAD_TIME + item_stats['safety_stock']
).round(2)

print('Reorder Point (ROP) computed')
print()
print('ROP stats:')
print(item_stats['rop'].describe().round(2))
print()
display(item_stats[['item','avg_daily_demand','std_daily_demand','safety_stock','rop']].head(10))

Reorder Point (ROP) computed

ROP stats:
count     50.00
mean     444.66
std      180.68
min      160.13
25%      260.35
50%      437.22
75%      590.61
max      744.70
Name: rop, dtype: float64



,item,avg_daily_demand,std_daily_demand,safety_stock,rop
0,1,21.981599,8.468922,36.86,190.73
1,2,58.574151,20.093015,87.44,497.46
2,3,36.642223,13.179441,57.36,313.86
3,4,22.010241,8.403898,36.57,190.64
4,5,18.358708,7.265167,31.62,160.13
5,6,58.503888,20.174898,87.80,497.33
6,7,58.531051,20.146002,87.67,497.39
7,8,76.950055,26.130697,113.72,652.37
8,9,51.389869,17.790158,77.42,437.15
9,10,73.227437,24.823725,108.03,620.62


In [6]:
# Cell 6 — Economic Order Quantity (EOQ)
# EOQ = sqrt(2 * D * S / H)
# D = annual demand, S = ordering cost, H = holding cost per unit per year

ORDERING_COST = 50   # $ per order
HOLDING_COST  = 2    # $ per unit per year

item_stats['annual_demand'] = item_stats['avg_daily_demand'] * 365
item_stats['eoq'] = np.sqrt(
    (2 * item_stats['annual_demand'] * ORDERING_COST) / HOLDING_COST
).round(2)

# Orders per year
item_stats['orders_per_year'] = (item_stats['annual_demand'] / item_stats['eoq']).round(1)

# Total annual inventory cost
item_stats['annual_inv_cost'] = (
    (item_stats['annual_demand'] / item_stats['eoq']) * ORDERING_COST +
    (item_stats['eoq'] / 2) * HOLDING_COST
).round(2)

print(f'Ordering Cost per Order : ${ORDERING_COST}')
print(f'Holding Cost per Unit   : ${HOLDING_COST}/year')
print()
print('EOQ stats:')
print(item_stats['eoq'].describe().round(2))
print()
display(item_stats[['item','annual_demand','eoq','orders_per_year','annual_inv_cost']].head(10))

Ordering Cost per Order : $50
Holding Cost per Unit   : $2/year

EOQ stats:
count      50.00
mean      954.28
std       209.27
min       578.83
25%       743.35
50%       968.26
75%      1127.28
max      1267.50
Name: eoq, dtype: float64



,item,annual_demand,eoq,orders_per_year,annual_inv_cost
0,1,8023.283680,633.38,12.7,1266.75
1,2,21379.565170,1033.91,20.7,2067.83
2,3,13374.411555,817.75,16.4,1635.51
3,4,8033.737952,633.79,12.7,1267.58
4,5,6700.928258,578.83,11.6,1157.66
5,6,21353.919222,1033.29,20.7,2066.59
6,7,21363.833790,1033.53,20.7,2067.07
7,8,28086.769989,1185.05,23.7,2370.10
8,9,18757.302026,968.43,19.4,1936.87
9,10,26728.014513,1156.03,23.1,2312.06


In [7]:
# Cell 7 — Stockout Risk Analysis
# Stockout risk = % of days demand exceeded avg + safety stock

stockout_results = []

for item_id in demand['item'].unique():
    item_demand = demand[demand['item'] == item_id]['sales'].values
    stats_row   = item_stats[item_stats['item'] == item_id].iloc[0]
    threshold   = stats_row['avg_daily_demand'] + stats_row['safety_stock']
    stockout_days = np.sum(item_demand > threshold)
    stockout_pct  = stockout_days / len(item_demand) * 100
    stockout_results.append({
        'item':          item_id,
        'stockout_days': stockout_days,
        'stockout_risk': round(stockout_pct, 2)
    })

stockout_df = pd.DataFrame(stockout_results)
item_stats  = item_stats.merge(stockout_df, on='item')

high_risk = item_stats[item_stats['stockout_risk'] > 10]
print(f'Items with >10% stockout risk : {len(high_risk)}')
print(f'Avg stockout risk             : {item_stats["stockout_risk"].mean():.2f}%')
print(f'Max stockout risk             : {item_stats["stockout_risk"].max():.2f}%')
display(item_stats.sort_values('stockout_risk', ascending=False).head(10))

Items with >10% stockout risk : 0
Avg stockout risk             : 0.01%
Max stockout risk             : 0.03%


,item,avg_daily_demand,std_daily_demand,max_daily_demand,min_daily_demand,total_sales,days_active,cv,safety_stock,rop,annual_demand,eoq,orders_per_year,annual_inv_cost,stockout_days,stockout_risk
22,23,29.297864,10.819549,81,3,534979,18260,0.369295,47.09,252.18,10693.720427,731.22,14.6,1462.44,5,0.03
40,41,22.002136,8.402470,60,2,401759,18260,0.381893,36.57,190.58,8030.779573,633.67,12.7,1267.34,3,0.02
41,42,36.688116,13.215112,96,5,669925,18260,0.360201,57.51,314.33,13391.162377,818.27,16.4,1636.53,3,0.02
29,30,40.337021,14.363331,115,5,736554,18260,0.356083,62.51,344.87,14723.012596,857.99,17.2,1715.98,3,0.02
30,31,58.644304,20.104705,159,10,1070845,18260,0.342825,87.49,498.00,21405.171139,1034.53,20.7,2069.07,4,0.02
33,34,25.735761,9.617910,79,2,469935,18260,0.373718,41.86,222.01,9393.552848,685.33,13.7,1370.66,3,0.02
34,35,65.801807,22.461990,168,12,1201541,18260,0.341358,97.75,558.36,24017.659639,1095.85,21.9,2191.70,3,0.02
20,21,40.317087,14.338006,109,7,736190,18260,0.355631,62.40,344.62,14715.736583,857.78,17.2,1715.56,3,0.02
46,47,22.003341,8.420102,61,2,401781,18260,0.382674,36.64,190.66,8031.219332,633.69,12.7,1267.38,3,0.02
3,4,22.010241,8.403898,66,0,401907,18260,0.381818,36.57,190.64,8033.737952,633.79,12.7,1267.58,4,0.02


In [8]:
# Cell 8 — Inventory Cost Savings (Before vs After EOQ)
# Before EOQ: assume ordering every 30 days
BASELINE_ORDER_FREQ = 12  # 12 times per year (monthly)

item_stats['baseline_cost'] = (
    BASELINE_ORDER_FREQ * ORDERING_COST +
    (item_stats['annual_demand'] / BASELINE_ORDER_FREQ / 2) * HOLDING_COST
).round(2)

item_stats['cost_savings']  = (item_stats['baseline_cost'] - item_stats['annual_inv_cost']).round(2)
item_stats['savings_pct']   = (item_stats['cost_savings'] / item_stats['baseline_cost'] * 100).round(1)

total_baseline = item_stats['baseline_cost'].sum()
total_eoq_cost = item_stats['annual_inv_cost'].sum()
total_savings  = item_stats['cost_savings'].sum()
savings_pct    = total_savings / total_baseline * 100

print('=== Inventory Cost Savings (EOQ vs Baseline) ===')
print(f'Total Baseline Cost : ${total_baseline:,.2f}/year')
print(f'Total EOQ Cost      : ${total_eoq_cost:,.2f}/year')
print(f'Total Savings       : ${total_savings:,.2f}/year')
print(f'Savings %           : {savings_pct:.1f}%')

=== Inventory Cost Savings (EOQ vs Baseline) ===
Total Baseline Cost : $109,463.95/year
Total EOQ Cost      : $95,427.96/year
Total Savings       : $14,035.99/year
Savings %           : 12.8%


In [9]:
# Cell 9 — Forecast-Based Demand Planning
# Use best forecast model predictions for next 90 days demand planning

# Determine best model column
best_model_name = models.sort_values('RMSE').iloc[0]['model'].lower()
col_map = {'sarima': 'sarima_pred', 'ets': 'ets_pred',
           'prophet': 'prophet_pred', 'xgboost': 'xgb_pred'}
best_col = col_map.get(best_model_name, 'prophet_pred')

# Fallback if column missing
if best_col not in forecast.columns:
    best_col = [c for c in forecast.columns if 'pred' in c][0]

forecast['best_forecast'] = forecast[best_col].fillna(forecast['sales'])

print(f'Using forecast column : {best_col}')
print(f'Forecast period       : {forecast["date"].min().date()} → {forecast["date"].max().date()}')
print(f'Avg daily forecast    : {forecast["best_forecast"].mean():,.2f}')
print(f'Total forecasted demand: {forecast["best_forecast"].sum():,.0f} units')

Using forecast column : xgb_pred
Forecast period       : 2017-10-03 → 2017-12-31
Avg daily forecast    : 27,249.67
Total forecasted demand: 2,452,470 units


In [10]:
# Cell 10 — Forecast vs Actual with Stockout Threshold
daily_avg    = demand.groupby('date')['sales'].sum().reset_index()
overall_avg  = daily_avg['sales'].mean()
overall_std  = daily_avg['sales'].std()
rop_overall  = overall_avg * LEAD_TIME + Z_SCORE * overall_std * np.sqrt(LEAD_TIME)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=forecast['date'], y=forecast['sales'],
    name='Actual', line=dict(color='#1a1a2e', width=2)))
fig.add_trace(go.Scatter(
    x=forecast['date'], y=forecast['best_forecast'],
    name=f'Forecast ({best_col})', line=dict(color='#1a936f', width=2, dash='dot')))
fig.add_hline(
    y=rop_overall, line_dash='dash', line_color='#e74c3c', line_width=1.5,
    annotation_text=f'Reorder Point ({rop_overall:.0f})',
    annotation_position='top right')

fig.update_layout(
    title='Forecast vs Actual with Reorder Point Threshold',
    xaxis_title='Date', yaxis_title='Total Daily Sales',
    height=460, plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#f0f0f0'),
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

In [11]:
# Cell 11 — Safety Stock vs ROP by Item
top_items = item_stats.sort_values('total_sales', ascending=False).head(20)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Safety Stock', x=top_items['item'].astype(str),
    y=top_items['safety_stock'], marker_color='#3498db'))
fig.add_trace(go.Bar(
    name='Reorder Point', x=top_items['item'].astype(str),
    y=top_items['rop'], marker_color='#e74c3c'))

fig.update_layout(
    barmode='group',
    title='Safety Stock & Reorder Point — Top 20 Items',
    xaxis_title='Item ID', yaxis_title='Units',
    height=420, plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#f0f0f0'),
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

In [12]:
# Cell 12 — EOQ vs Baseline Cost Comparison
top_cost = item_stats.sort_values('baseline_cost', ascending=False).head(15)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Baseline Cost', x=top_cost['item'].astype(str),
    y=top_cost['baseline_cost'], marker_color='#e74c3c',
    text=top_cost['baseline_cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside'))
fig.add_trace(go.Bar(
    name='EOQ Cost', x=top_cost['item'].astype(str),
    y=top_cost['annual_inv_cost'], marker_color='#1a936f',
    text=top_cost['annual_inv_cost'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside'))

fig.update_layout(
    barmode='group',
    title=f'Baseline vs EOQ Annual Inventory Cost — Total Savings: ${total_savings:,.0f} ({savings_pct:.1f}%)',
    xaxis_title='Item ID', yaxis_title='Annual Cost ($)',
    height=450, plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(showgrid=False), yaxis=dict(gridcolor='#f0f0f0', showticklabels=False),
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

In [13]:
# Cell 13 — Stockout Risk Distribution
fig1 = px.histogram(item_stats, x='stockout_risk', nbins=20,
                    title='Stockout Risk Distribution Across Items',
                    labels={'stockout_risk': 'Stockout Risk (%)'},
                    color_discrete_sequence=['#e74c3c'])
fig1.add_vline(x=10, line_dash='dash', line_color='#1a1a2e',
               annotation_text='10% threshold', annotation_position='top right')
fig1.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white')
fig1.show()

fig2 = px.scatter(item_stats, x='avg_daily_demand', y='stockout_risk',
                  size='safety_stock', color='stockout_risk',
                  color_continuous_scale='RdYlGn_r',
                  title='Demand vs Stockout Risk (bubble = safety stock size)',
                  labels={'avg_daily_demand': 'Avg Daily Demand',
                          'stockout_risk': 'Stockout Risk (%)'},
                  hover_data=['item'])
fig2.update_layout(height=420, plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

In [14]:
# Cell 14 — Inventory Recommendations Table
def get_recommendation(row):
    if row['stockout_risk'] > 15:
        return '🔴 URGENT — Increase safety stock immediately'
    elif row['stockout_risk'] > 10:
        return '🟡 WARNING — Review reorder point'
    elif row['cv'] > 0.3:
        return '🟠 MONITOR — High demand variability'
    elif row['savings_pct'] > 20:
        return '💰 OPTIMIZE — Switch to EOQ ordering'
    else:
        return '🟢 HEALTHY — No action needed'

item_stats['recommendation'] = item_stats.apply(get_recommendation, axis=1)

reco_table = item_stats[[
    'item', 'avg_daily_demand', 'safety_stock', 'rop',
    'eoq', 'stockout_risk', 'cost_savings', 'recommendation'
]].sort_values('stockout_risk', ascending=False)

reco_table.columns = [
    'Item', 'Avg Daily Demand', 'Safety Stock',
    'Reorder Point', 'EOQ', 'Stockout Risk (%)',
    'Annual Savings ($)', 'Recommendation'
]

print('=== Inventory Recommendations ===')
display(reco_table.head(20))

=== Inventory Recommendations ===


,Item,Avg Daily Demand,Safety Stock,Reorder Point,EOQ,Stockout Risk (%),Annual Savings ($),Recommendation
22,23,29.297864,47.09,252.18,731.22,0.03,28.70,🟠 MONITOR — High demand variability
40,41,22.002136,36.57,190.58,633.67,0.02,1.89,🟠 MONITOR — High demand variability
41,42,36.688116,57.51,314.33,818.27,0.02,79.40,🟠 MONITOR — High demand variability
29,30,40.337021,62.51,344.87,857.99,0.02,110.94,🟠 MONITOR — High demand variability
30,31,58.644304,87.49,498.00,1034.53,0.02,314.69,🟠 MONITOR — High demand variability
33,34,25.735761,41.86,222.01,685.33,0.02,12.14,🟠 MONITOR — High demand variability
34,35,65.801807,97.75,558.36,1095.85,0.02,409.77,🟠 MONITOR — High demand variability
20,21,40.317087,62.40,344.62,857.78,0.02,110.75,🟠 MONITOR — High demand variability
46,47,22.003341,36.64,190.66,633.69,0.02,1.89,🟠 MONITOR — High demand variability
3,4,22.010241,36.57,190.64,633.79,0.02,1.90,🟠 MONITOR — High demand variability


In [15]:
# Cell 15 — Recommendation Summary
reco_counts = item_stats['recommendation'].value_counts()

fig = px.pie(values=reco_counts.values,
             names=reco_counts.index,
             title='Inventory Health — Recommendation Breakdown',
             hole=0.42,
             color_discrete_sequence=['#e74c3c','#f39c12','#e67e22','#3498db','#1a936f'])
fig.update_traces(textposition='inside', textinfo='percent+label', textfont_size=11)
fig.update_layout(height=380, showlegend=False,
                  paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
fig.show()

In [16]:
# Cell 16 — Save All Results
import os
os.makedirs('../data/processed', exist_ok=True)

item_stats.to_csv('../data/processed/supply_chain_results.csv', index=False)
reco_table.to_csv('../data/processed/inventory_recommendations.csv', index=False)

print('supply_chain_results.csv saved')
print('inventory_recommendations.csv saved')
print()
print('=' * 48)
print('        FINAL PROJECT SUMMARY')
print('=' * 48)
print(f'Total items analyzed      : {len(item_stats)}')
print(f'Avg safety stock          : {item_stats["safety_stock"].mean():.2f} units')
print(f'Avg reorder point         : {item_stats["rop"].mean():.2f} units')
print(f'Avg EOQ                   : {item_stats["eoq"].mean():.2f} units')
print(f'Avg stockout risk         : {item_stats["stockout_risk"].mean():.2f}%')
print(f'High risk items (>10%)    : {len(high_risk)}')
print(f'Total inventory savings   : ${total_savings:,.2f}/year')
print(f'Savings %                 : {savings_pct:.1f}%')
print(f'Best forecast model       : {best["model"]} (RMSE: {best["RMSE"]})')
print('=' * 48)

supply_chain_results.csv saved
inventory_recommendations.csv saved

        FINAL PROJECT SUMMARY
Total items analyzed      : 50
Avg safety stock          : 78.91 units
Avg reorder point         : 444.66 units
Avg EOQ                   : 954.28 units
Avg stockout risk         : 0.01%
High risk items (>10%)    : 0
Total inventory savings   : $14,035.99/year
Savings %                 : 12.8%
Best forecast model       : XGBoost (RMSE: 895.95)
